In [76]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Allow the notebook to import modules from the project root.
sys.path.append("..")

from config import CONFIG, REQUIRED_COLUMNS
from src.data_processing import (
    load_csv_chunks,
    prepare_chunk,
)
from src.analysis import CustomsAnalyzer
from src.numpy_analysis import run_numpy_comparison
from src.visualization import (
    create_bar_plot,
    create_heatmap,
)
from src.validation import (
    create_validation_results,
    save_validation_results,
)

In [77]:
chunks = load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
)

first_chunk = next(chunks)

print("First chunk rows:", len(first_chunk))
print("Columns:")
print(first_chunk.columns.tolist())

First chunk rows: 100000
Columns:
['tq', 'dutiablevaluephp', 'countryorigin_iso3']


In [78]:
analyzer = CustomsAnalyzer(
    output_dir=CONFIG["output_dir"],
    group_columns=CONFIG["group_columns"],
    measure_column=CONFIG["measure_column"],
)

chunks = load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
)

for chunk in chunks:
    selected = prepare_chunk(
        chunk,
        CONFIG["minimum_dutiable_value_php"],
        CONFIG["high_value_threshold_million_php"],
    )

    analyzer.add_chunk(
        selected,
        len(chunk),
        len(selected),
    )

data = analyzer.combine_chunks()

print("Selected rows:", len(data))
print()
print(data.head())

Selected rows: 2236612

       tq  dutiablevaluephp countryorigin_iso3  dutiablevalue_million_php  \
0  2015q1          90088007                MYS                  90.088007   
1  2015q1          15742304                CHN                  15.742304   
2  2015q1          52605886                CHN                  52.605886   
3  2015q1          34181694                CHN                  34.181694   
4  2015q1           7522344                KOR                   7.522344   

  value_band  
0   Standard  
1   Standard  
2   Standard  
3   Standard  
4   Standard  


In [79]:
data.loc[
    (
        data["tq"].notna()
        & (data["dutiablevaluephp"] > 0)
    ),
    [
        "tq",
        "countryorigin_iso3",
        "dutiablevaluephp",
        "dutiablevalue_million_php",
        "value_band",
    ],
].head(10)

,tq,countryorigin_iso3,dutiablevaluephp,dutiablevalue_million_php,value_band
0,2015q1,MYS,90088007,90.088007,Standard
1,2015q1,CHN,15742304,15.742304,Standard
2,2015q1,CHN,52605886,52.605886,Standard
3,2015q1,CHN,34181694,34.181694,Standard
4,2015q1,KOR,7522344,7.522344,Standard
5,2015q1,KOR,937401,0.937401,Standard
6,2015q1,KOR,454267,0.454267,Standard
7,2015q1,KOR,437421,0.437421,Standard
8,2015q1,CHN,9536863,9.536863,Standard
9,2015q1,CHN,123198882,123.198882,High


In [80]:
grouped = analyzer.create_grouped_summary(data)

grouped.head(10)

,countryorigin_iso3,row_count,valid_measure_count,measure_sum,measure_mean
0,AFG,5,5,588546,1.177092e+05
1,AGO,4,4,11752842,2.938210e+06
2,AIA,26,26,6120010,2.353850e+05
3,ALB,33,33,4355770,1.319930e+05
4,AND,8,8,16228840,2.028605e+06
5,ANT,81,81,393863146,4.862508e+06
6,ARE,2851,2851,29645763789,1.039837e+07
7,ARG,1089,1089,13213323494,1.213345e+07
8,ARM,5,5,998542,1.997084e+05
9,ASM,14,14,2575220,1.839443e+05


In [81]:
print("Number of country groups:", len(grouped))
print("Total grouped rows:", grouped["row_count"].sum())
print("Total grouped measure:", grouped["measure_sum"].sum())

Number of country groups: 197
Total grouped rows: 2236612
Total grouped measure: 3587267375257


In [82]:
grouped.to_csv(
    CONFIG["output_dir"] / "grouped.csv",
    index=False,
)

print("grouped.csv saved.")

grouped.csv saved.


In [83]:
grouped_two = analyzer.create_two_category_summary(data)

grouped_two.head(10)

,countryorigin_iso3,tq,row_count,measure_sum
0,AFG,2015q3,3,567713
1,AFG,2015q4,2,20833
2,AGO,2015q3,4,11752842
3,AIA,2015q3,9,1429763
4,AIA,2015q4,17,4690247
5,ALB,2015q2,1,138212
6,ALB,2015q3,16,947695
7,ALB,2015q4,16,3269863
8,AND,2015q2,3,1551437
9,AND,2015q3,3,4748724


In [84]:
print("Number of country-quarter groups:", len(grouped_two))
print("Total grouped rows:", grouped_two["row_count"].sum())
print("Total grouped measure:", grouped_two["measure_sum"].sum())

Number of country-quarter groups: 628
Total grouped rows: 2236612
Total grouped measure: 3587267375257


In [85]:
grouped_two.to_csv(
    CONFIG["output_dir"] / "grouped_two.csv",
    index=False,
)

print("grouped_two.csv saved.")

grouped_two.csv saved.


In [86]:
pivot = analyzer.create_pivot(grouped_two)

pivot.head(10)

tq,countryorigin_iso3,2015q1,2015q2,2015q3,2015q4,Total
0,AFG,NaN,NaN,5.677130e+05,2.083300e+04,588546
1,AGO,NaN,NaN,1.175284e+07,NaN,11752842
2,AIA,NaN,NaN,1.429763e+06,4.690247e+06,6120010
3,ALB,NaN,1.382120e+05,9.476950e+05,3.269863e+06,4355770
4,AND,NaN,1.551437e+06,4.748724e+06,9.928679e+06,16228840
5,ANT,4.093564e+07,3.245682e+07,2.156041e+08,1.048666e+08,393863146
6,ARE,1.143155e+10,5.646697e+09,8.342975e+09,4.224541e+09,29645763789
7,ARG,2.034266e+09,8.810729e+08,3.759537e+09,6.538448e+09,13213323494
8,ARM,NaN,NaN,2.677720e+05,7.307700e+05,998542
9,ASM,3.325130e+05,NaN,6.254110e+05,1.617296e+06,2575220


In [87]:
print("Pivot grand total:", pivot.loc[pivot["countryorigin_iso3"] == "Total", "Total"].iloc[0])

Pivot grand total: 3587267375257


In [88]:
pivot.to_csv(
    CONFIG["output_dir"] / "pivot.csv",
    index=False,
)

print("pivot.csv saved.")

pivot.csv saved.


In [89]:
top10 = analyzer.create_top10(grouped)

top10

,countryorigin_iso3,row_count,valid_measure_count,measure_sum,measure_mean
0,CHN,589626,589626,651605135422,1.105116e+06
1,JPN,352375,352375,304562507423,8.643136e+05
2,USA,209259,209259,291521135474,1.393112e+06
3,KOR,112063,112063,266807274024,2.380869e+06
4,THA,86863,86863,252139451874,2.902726e+06
5,TWN,93071,93071,244973287949,2.632112e+06
6,SGP,208349,208349,230099848496,1.104396e+06
7,IDN,35819,35819,162933845447,4.548811e+06
8,MYS,64880,64880,161741895330,2.492939e+06
9,SAU,602,602,158896912371,2.639484e+08


In [90]:
print("Number of rows in top10:", len(top10))
print("Top 10 total measure:", top10["measure_sum"].sum())

Number of rows in top10: 10
Top 10 total measure: 2725281293810


In [91]:
top10.to_csv(
    CONFIG["output_dir"] / "top10.csv",
    index=False,
)

print("top10.csv saved.")

top10.csv saved.


In [92]:
numpy_results = run_numpy_comparison(data)

numpy_results

,method,result,median_time_seconds
0,Python loop,1.627477e+10,0.001420
1,NumPy vectorized,1.627477e+10,0.000015


In [93]:
print("Loop and vectorized results agree:", numpy_results.attrs["numpy_equal"])

Loop and vectorized results agree: True


In [94]:
numpy_results.to_csv(
    CONFIG["output_dir"] / "numpy_comparison.csv",
    index=False,
)

print("numpy_comparison.csv saved.")

numpy_comparison.csv saved.


In [95]:
create_bar_plot(
    top10,
    CONFIG["output_dir"] / "bar.png",
)

print("bar.png created.")

bar.png created.


In [96]:
create_heatmap(
    pivot,
    CONFIG["output_dir"] / "heatmap.png",
)

print("heatmap.png created.")

heatmap.png created.


In [97]:
audit_log = pd.DataFrame(analyzer.audit_records)

audit_log

,step,operation,rule,rows_before,rows_after
0,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
1,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
2,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
3,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
4,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
5,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
6,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
7,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
8,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000
9,filter,Apply two-condition filter,tq is not missing AND dutiablevaluephp > 0,100000,100000


In [98]:
audit_log.to_csv(
    CONFIG["output_dir"] / "audit_log.csv",
    index=False,
)

print("audit_log.csv saved.")

audit_log.csv saved.
